In [ ]:
# Library Management
import csv
import json

class BookNotFoundException(Exception):
    pass

class BookAlreadyBorrowedException(Exception):
    pass

class MemberLimitExceededException(Exception):
    pass

class NotEnoughCredentialsException(Exception):
    pass

class NameErrorException(Exception):
    pass

class MemberNotFoundException(Exception):
    pass


class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author
        self.is_borrowed = False

    def __str__(self):
        return f"Book({self.title}, {self.author})"
    
class Member:
    def __init__(self, name):
        self.name = name
        self.borrowed_books = []

    def can_borrow(self):
        return len(self.borrowed_books) < 3

class Library:
    def __init__(self):
        self.books = []
        self.members = {}

    def add_book(self, title, author):
        if not title:
            raise NotEnoughCredentialsException("Title not found")

        if not author:
            raise NotEnoughCredentialsException("Author not found")
        book = Book(title, author)
        available = "Available" if not book.is_borrowed else "Not available"
        # new_book = [book.title, book.author, book.is_borrowed]
        self.books.append(book)
        # print(f"Book successfully added!: {new_book[0]}, {new_book[1]}, {new_book[2]}")
        print(f"Book successfully added!: {book.title}, {book.author}, {available}\n")

    def add_member(self, name):
        if not name:
            raise NameErrorException("Name not found")
        member = Member(name)
        self.members[name] = member
        print(f"Member added: {name}\n")
    def borrow_book(self, member_name, title):
        if member_name not in self.members:
            raise MemberNotFoundException("Member not found")
        member = self.members[member_name]

        book = self.find_book(title)

        if book.is_borrowed:
            raise BookAlreadyBorrowedException("Book is already borrowed")

        if not member.can_borrow():
            raise MemberLimitExceededException("Member has reached the borrowing limit")

        book.is_borrowed = True
        member.borrowed_books.append(book.title)
        print(f"{member_name}, successfully borrowed '{book.title}'\n")


    def return_book(self, member_name, title):
        if member_name not in self.members:
            raise MemberNotFoundException("Member not found")
        member = self.members[member_name]

        titles_lower = [t.lower() for t in member.borrowed_books]
        if title.lower() not in titles_lower:
            raise BookNotFoundException("User does not have this book")

        index = titles_lower.index(title.lower())
        actual_title = member.borrowed_books[index]

        book = self.find_book(actual_title)
        book.is_borrowed = False
        member.borrowed_books.remove(actual_title)
        print(f"{member_name}, successfully returned '{book.title}'\n")

    def view_all(self):
        if not self.books:
            raise BookNotFoundException("Books not found")
        else:
            for book in self.books:
                available = "Available" if not book.is_borrowed else "Not available"
                print(f"{book.title}, {book.author}, {available}")

    def find_book(self, title):
        books = self.books
        for book in books:
            available = "Available" if not book.is_borrowed else "Not available"
            if book.title.lower() == title.lower():
                print(f"{book.title}, {book.author}, {available}")
                return book
        raise BookNotFoundException("Book not found")        
    def save(self):
        columns = ["Title", "Author", "Status"]
        data = self.books
        if not self.books:
            print("Books not found\n")
            return
        with open('library.csv', mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(columns)
            writer.writerows([b.title, b.author, b.is_borrowed] for b in data)
        print("Successfully saved\n")

    def load(self):
        self.books = []
        try:
            with open('library.csv', 'rt') as file:
                reader = csv.reader(file)
                columns = next(reader)
                for row in reader:
                    book = Book(row[0], row[1])
                    book.is_borrowed = (row[2] == "True")
                    self.books.append(book)
            print("Successfully loaded\n")
        except FileNotFoundError:
            print("No saved file found, starting fresh.\n")

    def save_members(self):
        if not self.members:
            print("No members to save\n")
            return
        data = {}
        for name, member in self.members.items():
            data[name] = member.borrowed_books
        with open('members.json', 'w') as f:
            json.dump(data, f, indent=2)
        print("Members saved successfully\n")

    def load_members(self):
        self.members = {}
        try:
            with open('members.json', 'r') as f:
                data = json.load(f)
            for name, borrowed_books in data.items():
                member = Member(name)
                member.borrowed_books = borrowed_books
                self.members[name] = member
            print("Members loaded successfully\n")
        except FileNotFoundError:
            print("No saved members file found, starting fresh.\n")

    def run(self):
        self.load()
        self.load_members()
        while True:
            print(f"1. Add book\n2. Add member\n3. Borrow book\n4. Return book\n5. View all books\n6. Find a book\n7. Save\n8. Load\n9. Exit")
            choice = input("Select: ")
            if choice not in ['1','2','3','4','5','6','7','8','9']:
                print(f"Not command found with {choice}")
                continue
            try:
                if choice == "9":
                    break
                elif choice == "1":
                    title = input("Title: ")
                    author = input("Author: ")
                    self.add_book(title, author)
                elif choice == "2":
                    member = input("Member name: ")
                    self.add_member(member)
                elif choice == "3":
                    member = input("Member name: ")
                    title = input("Title name: ")
                    self.borrow_book(member, title)
                elif choice == "4":
                    member = input("Member name: ")
                    title = input("Title name: ")
                    self.return_book(member, title)
                elif choice == "5":
                    self.view_all()
                elif choice == "6":
                    title = input("Searching title: ")
                    self.find_book(title)
                elif choice == "7":
                    self.save()
                    self.save_members()
                elif choice == "8":
                    self.load()
                    self.load_members()
            except (BookNotFoundException, NotEnoughCredentialsException, NameErrorException, MemberNotFoundException, BookAlreadyBorrowedException, MemberLimitExceededException) as e:
                print(f"Error: {e}\n")
            print('\n')

manager_library = Library()
manager_library.run()


0
1
2


In [ ]:
# Grade Management
import csv

class CommanDNotFound(Exception):
    pass

class NotEnoughCredentials(Exception):
    pass

class FIleNotFoundError(Exception):
    pass

class Student:
    def __init__(self, name, subject):
        self.name = name
        self.subject = subject
        self.grade = 0

class Grade:
    def __init__(self):
        self.data = []
        self.averages = {}

    def grade(self, name, subject, grade):
        if not (name and subject and grade):
            raise NotEnoughCredentials("Not enough credentials")
        data = self.data
        student = Student(name, subject)
        student.grade = grade
        data.append({"Name": student.name, "Subject": student.subject, "Grade": student.grade})
        print(f"Student successfully graded")      

    def read(self):
        data = self.data
        if not data:
            print("Data not found\n")
            return
        
        for student in data:
            print(f"{student["Name"]}, {student["Subject"]}, {student["Grade"]}")
    def average_grades(self):
        data = self.data
        all_grades = {}
        for sub in data:
            subject = sub["Subject"]
            grade = float(sub["Grade"])
            if subject not in all_grades:
                all_grades[subject] = []
            all_grades[subject].append(grade)

        for subject in all_grades:
            grade_list = all_grades[subject]
            average = sum(grade_list) / len(grade_list)
            self.averages[subject] = average
            print(f"{subject}: {average}")
    def save(self):
        data = self.data
        if not data:
            print("Data not found")
            return
        columns = ["Name", "Subject", "Grade"]
        with open("grades.csv", mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(columns)
            writer.writerows([s["Name"], s["Subject"], s["Grade"]] for s in data)
        print("Data successfully saved!\n")

    def load(self):
        self.data = []
        try:
            with open("grades.csv", "rt") as file:
                reader = csv.reader(file)
                columns = next(reader)
                for row in reader:
                    self.data.append({"Name":row[0], "Subject":row[1], "Grade":row[2]})
                # print(reader)
        except FileNotFoundError:
            raise FIleNotFoundError("Error in file loading")
        print("Data successfully loaded")

    def save_averages(self):
        averages = self.averages
        if not averages:
            print("Data not found")
            return
        columns = ["Subject", "Average Grade"]
        with open("average_grades.csv", mode='wt', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(columns)
            writer.writerows([[k, v] for k, v in zip(averages.keys(), averages.values())])
        print("Average Grades successfully saved!")

    def load_averages(self):
        self.averages = {}
        try:
            with open("average_grades.csv", mode='rt') as file:
                reader = csv.reader(file)
                columns = next(reader)
                for row in reader:
                    self.averages[row[0]] = row[1]
        except FileNotFoundError:
            raise FIleNotFoundError("Error in file loading")
    def run(self):
        self.load()
        self.load_averages()
        while(True):
            print("\n")
            print(f"1. Grade a student\n2. Read\n3. Average grades\n4. Save\n5. Load\n6. Exit")
            choice = input("Select: ")
            try:
                if choice not in ["1","2","3","4","5","6",]:
                    raise CommanDNotFound(f"Command not found, with {choice}")

                if choice == "6":
                    break
                elif choice == "1":
                    name = input("Name: ")
                    subject = input("Subject: ")
                    grade = input("Grade: ")
                    self.grade(name, subject, grade)
                elif choice == "2":
                    self.read()
                elif choice == "3":
                    self.average_grades()
                elif choice == "4":
                    self.save()
                    self.save_averages()
                elif choice == "5":
                    self.load()
                    self.load_averages()
            except (CommanDNotFound, FileNotFoundError, NotEnoughCredentials) as e:
                print(f"Error: {e}\n")
grading_manager = Grade()
grading_manager.run()
            

In [ ]:
# Task Management
import json
import csv

class Task:
    def __init__(self, id, task, completed, priority):
        self.id = id
        self.task = task
        self.completed = completed
        self.priority = priority

class Manager:
    def __init__(self):
        self.tasks = []
        self.next_task_id = 1

    def create(self, task, completed_input, priority):
        try:
            task_id = self.next_task_id

            if not task:
                print("Error: Task name cannot be empty.")
                return

            if completed_input not in ["True", "False"]:
                print("You should enter 'True' or 'False' for 'completed' section")
                return

            if not isinstance(priority, int):
                print("Error: Priority must be a number.")
                return

            completed = True if completed_input == "True" else False
            task = Task(task_id, task, completed, priority)

            self.tasks.append({
                'id': task.id,
                'task': task.task,
                'completed': task.completed,
                'priority': task.priority
            })

            self.next_task_id += 1
            print(f"Task successfully created: id: {task.id}, task: {task.task}, completed: {task.completed}, priority: {task.priority}\n")

        except (TypeError, ValueError) as e:
            print(f"Error while creating task: {e}")

    def read(self):
        try:
            if not self.tasks:
                print("Tasks not found")
                return

            for task in self.tasks:
                print(f"{task['id']}, {task['task']}, {task['completed']}, {task['priority']}")

        except KeyError as e:
            print(f"Error: Missing field {e} in task data.")
        except TypeError as e:
            print(f"Error while reading tasks: {e}")

    def update(self, id, task, completed_input, priority):
        try:
            if completed_input not in ["True", "False", ""]:
                print("You should enter 'True' or 'False' for 'completed' section")
                return

            found = False

            if completed_input == "True":
                completed = True
            elif completed_input == "False":
                completed = False
            else:
                completed = None

            for row in self.tasks:
                if str(id) == str(row['id']):
                    row["task"] = task if task else row["task"]
                    row["completed"] = completed if completed is not None else row["completed"]
                    row["priority"] = priority if priority is not None else row["priority"]
                    found = True

            if not found:
                print(f"Task with {id} id not found")
            else:
                print(f"Task with {id} id successfully updated")

        except KeyError as e:
            print(f"Error: Missing field {e} in task data.")
        except (TypeError, ValueError) as e:
            print(f"Error while updating task: {e}")

    def delete(self, id):
        try:
            found = False

            for row in self.tasks:
                if str(row['id']) == str(id):
                    self.tasks.remove(row)
                    found = True
                    break

            if not found:
                print(f"Task with {id} id not found")
            else:
                print(f"Task with {id} id successfully deleted")

        except KeyError as e:
            print(f"Error: Missing field {e} in task data.")
        except (TypeError, ValueError) as e:
            print(f"Error while deleting task: {e}")

    def total_tasks(self):
        try:
            print(f"The total number of tasks: {len(self.tasks)}")
        except TypeError as e:
            print(f"Error while counting tasks: {e}")

    def completed_tasks(self):
        try:
            num = 0
            for row in self.tasks:
                if row['completed'] is True:
                    num += 1
            print(f"The number of completed tasks: {num}")

        except KeyError as e:
            print(f"Error: Missing field {e} in task data.")
        except TypeError as e:
            print(f"Error while counting completed tasks: {e}")

    def pending_tasks(self):
        try:
            num = 0
            for row in self.tasks:
                if row['completed'] is False:
                    num += 1
            print(f"The number of not completed tasks: {num}")

        except KeyError as e:
            print(f"Error: Missing field {e} in task data.")
        except TypeError as e:
            print(f"Error while counting pending tasks: {e}")

    def average_priority(self):
        try:
            if not self.tasks:
                print("Error: Cannot calculate average priority because there are no tasks.")
                return

            num = 0
            for row in self.tasks:
                num += int(row['priority'])

            res = num / len(self.tasks)
            print(f"The average priority level of all tasks: {res}")

        except KeyError as e:
            print(f"Error: Missing field {e} in task data.")
        except (TypeError, ValueError) as e:
            print(f"Error: Invalid priority value: {e}")
        except ZeroDivisionError:
            print("Error: Cannot divide by zero.")

    def save(self):
        try:
            with open("tasks.json", "wt") as file:
                json.dump(self.tasks, file, indent=2)
            print("Tasks successfully saved")

        except OSError as e:
            print(f"Error while saving tasks.json: {e}")
        except TypeError as e:
            print(f"Error: Data cannot be converted to JSON: {e}")

    def load(self):
        try:
            self.tasks = []

            with open("tasks.json", "rt") as file:
                reader = json.load(file)

            if not isinstance(reader, list):
                print("Error: tasks.json should contain a list of tasks.")
                return

            if not reader:
                print("Error: tasks.json is empty.")
                return

            for row in reader:
                if not isinstance(row, dict):
                    print("Error: Invalid task format in tasks.json.")
                    self.tasks = []
                    return

                required_fields = ["id", "task", "completed", "priority"]

                for field in required_fields:
                    if field not in row:
                        print(f"Error: Missing '{field}' field in tasks.json.")
                        self.tasks = []
                        return

                if not isinstance(row["id"], int):
                    print("Error: Task ID must be an integer.")
                    self.tasks = []
                    return

                if not isinstance(row["task"], str):
                    print("Error: Task name must be a string.")
                    self.tasks = []
                    return

                if not isinstance(row["completed"], bool):
                    print("Error: Completed must be True or False.")
                    self.tasks = []
                    return

                try:
                    int(row["priority"])
                except (TypeError, ValueError):
                    print("Error: Priority must be a number.")
                    self.tasks = []
                    return

                self.tasks.append(row)

            self.next_task_id = max(row["id"] for row in self.tasks) + 1
            print("Tasks successfully loaded!\n")

        except FileNotFoundError:
            print("Error: tasks.json file was not found.")
            print("Please create tasks.json first.")
        except json.JSONDecodeError:
            print("Error: tasks.json contains invalid JSON.")
        except PermissionError:
            print("Error: Permission denied while reading tasks.json.")
        except OSError as e:
            print(f"Error while reading tasks.json: {e}")
        except (KeyError, TypeError, ValueError) as e:
            print(f"Error: Invalid data in tasks.json: {e}")

    def convert_to_csv(self):
        try:
            if not self.tasks:
                print("Error: There are no tasks to convert.")
                return

            columns = ["ID", "Task Name", "Completed Status", "Priority"]

            with open("tasks.csv", "wt", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(columns)

                for task in self.tasks:
                    writer.writerow([
                        task["id"],
                        task["task"],
                        task["completed"],
                        task["priority"]
                    ])

            print("JSON file successfully converted to CSV file")

        except KeyError as e:
            print(f"Error: Missing field {e} in task data.")
        except PermissionError:
            print("Error: Permission denied while creating tasks.csv.")
        except OSError as e:
            print(f"Error while creating tasks.csv: {e}")
        except csv.Error as e:
            print(f"Error while writing CSV file: {e}")

    def run(self):
        self.load()

        while True:
            print("\n")
            print("1. Create a task\n2. Read tasks\n3. Update a task\n4. Delete a task\n5. Total tasks\n6. Completed tasks\n7. Pending tasks\n8. Average priority\n9. Save\n10. Load\n11. Convert to CSV file\n12. Exit")

            choice = input("Select: ")

            if choice not in ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"]:
                print("Command Not Found")
                continue

            if choice == "12":
                print("Program exited.")
                break

            elif choice == "1":
                try:
                    task = input("Task name: ")
                    completed_input = input("Completed(True/False): ")
                    priority = int(input("Priority: "))
                    self.create(task, completed_input, priority)
                except ValueError:
                    print("Error: Priority must be a number.")

            elif choice == "2":
                self.read()

            elif choice == "3":
                try:
                    id = input("Task id for updating: ")
                    task = input("Updated name: ")
                    completed_input = input("Change a completed section(True/False): ")
                    priority_input = input("Updated priority: ")

                    if priority_input == "":
                        priority = None
                    else:
                        priority = int(priority_input)

                    self.update(id, task, completed_input, priority)

                except ValueError:
                    print("Error: Priority must be a number.")

            elif choice == "4":
                try:
                    id = input("Deleting ID: ")

                    if not id:
                        print("Error: ID cannot be empty.")
                        continue

                    self.delete(id)

                except ValueError:
                    print("Error: Invalid ID.")

            elif choice == "5":
                self.total_tasks()

            elif choice == "6":
                self.completed_tasks()

            elif choice == "7":
                self.pending_tasks()

            elif choice == "8":
                self.average_priority()

            elif choice == "9":
                self.save()

            elif choice == "10":
                self.load()

            elif choice == "11":
                self.convert_to_csv()


task_manager = Manager()
task_manager.run()
